# Simulasi FSM

Notebook ini digunakan untuk mengecek apakah string merupakan anggota bahasa:

**L = { x ∈ (0 + 1)+ | karakter terakhir pada string x adalah 1 dan x tidak memiliki substring 00 }**

Fitur:
- Diagram FSD/FSM
- Input string biner
- Tombol **Prev / Next / Reset** untuk simulasi per langkah
- State aktif dan transisi aktif disorot
- Keterangan hasil akhir diterima / ditolak
- Alasan mengapa ditolak


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.ioff()


In [ ]:
FINAL_STATE = 'B'

positions = {
    'S': (0.0, 0.0),
    'A': (2.5, 1.55),
    'B': (2.5, -1.55),
    'C': (4.95, 1.55),
}

state_desc = {
    'S': 'Start State',
    'A': 'Terakhir membaca 0',
    'B': 'Terakhir membaca 1 (Final State)',
    'C': 'Trap State',
}

transitions = {
    ('S', '0'): 'A',
    ('S', '1'): 'B',
    ('A', '0'): 'C',
    ('A', '1'): 'B',
    ('B', '0'): 'A',
    ('B', '1'): 'B',
    ('C', '0'): 'C',
    ('C', '1'): 'C',
}

edge_specs = [
    ('S', 'A', '0', 'diag_up'),
    ('S', 'B', '1', 'diag_down'),
    ('A', 'C', '0', 'right'),
    ('A', 'B', '1', 'vertical_down_right'),
    ('B', 'A', '0', 'vertical_up_left'),
    ('B', 'B', '1', 'self_bottom_right'),
    ('C', 'C', '0,1', 'self_top_right'),
]

def valid_binary(text):
    if len(text) == 0:
        return False
    for ch in text:
        if ch != '0' and ch != '1':
            return False
    return True

def simulate_steps(text):
    steps = []
    state = 'S'

    steps.append({
        'step': 0,
        'symbol': None,
        'from_state': None,
        'to_state': state,
        'remaining': text,
        'log': 'State awal = S'
    })

    found_double_zero = False

    for i, ch in enumerate(text):
        prev_state = state

        if state == 'S':
            if ch == '0':
                state = 'A'
            else:
                state = 'B'
        elif state == 'A':
            if ch == '0':
                state = 'C'
                found_double_zero = True
            else:
                state = 'B'
        elif state == 'B':
            if ch == '0':
                state = 'A'
            else:
                state = 'B'
        elif state == 'C':
            state = 'C'

        steps.append({
            'step': i + 1,
            'symbol': ch,
            'from_state': prev_state,
            'to_state': state,
            'remaining': text[i + 1:],
            'log': f"Dibaca {ch}, current state = {state}",
            'detail': f"state {prev_state} -> {state}"
        })

    accepted = (state == 'B')

    if accepted:
        result = f'String {text} dikenali oleh FSD'
        reason = 'String berakhir dengan 1 dan tidak memiliki substring 00'
    else:
        result = f'String {text} tidak dikenali oleh FSD'
        if found_double_zero:
            reason = 'String memiliki substring 00'
        elif text[-1] != '1':
            reason = 'Karakter terakhir bukan 1'
        else:
            reason = 'String tidak memenuhi aturan bahasa'

    return steps, accepted, result, reason


In [ ]:
from matplotlib.patches import Circle, FancyArrowPatch

def draw_state(ax, name, pos, active=False, final=False):
    x, y = pos

    face = '#f3f3f3'

    outer = Circle((x, y), radius=0.42, facecolor=face, edgecolor='black', lw=0.9, zorder=5)
    ax.add_patch(outer)

    # active ring
    if active:
        ring_color = '#e53935'
        ring = Circle((x, y), radius=0.47, fill=False, edgecolor=ring_color, lw=1.4, zorder=6)
        ax.add_patch(ring)

    # final state inner circle
    if final:
        inner = Circle((x, y), radius=0.34, fill=False, edgecolor='#304ffe', lw=1.0, zorder=6)
        ax.add_patch(inner)

    ax.text(x, y, name, ha='center', va='center', fontsize=18, fontweight='normal', zorder=7)


def draw_self_loop(ax, state, label, mode='self_top_right', active=False):
    x, y = positions[state]
    color = '#d32f2f' if active else 'black'
    lw = 2.0 if active else 1.5

    if mode == 'self_bottom_right':
        # LOOP B: jelas di kanan bawah node B
        start = (x + 0.18, y - 0.36)
        end   = (x - 0.02, y - 0.34)
        rad = -2.2
        text_xy = (x + 0.62, y - 0.72)
    else:
        # LOOP C: jelas di kanan atas node C, tidak nabrak label
        start = (x + 0.10, y + 0.34)
        end   = (x + 0.34, y + 0.08)
        rad = -2.0
        text_xy = (x + 0.72, y + 0.20)

    loop = FancyArrowPatch(
        start, end,
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle='-|>',
        mutation_scale=16,
        lw=lw,
        color=color,
        fill=False,
        zorder=3
    )
    ax.add_patch(loop)
    ax.text(text_xy[0], text_xy[1], label, fontsize=16, color=color,
            ha='center', va='center', zorder=8)


def draw_edge(ax, src, dst, label, mode='right', active=False):
    color = '#d32f2f' if active else 'black'
    lw = 2.0 if active else 1.5

    if mode in ['self_bottom_right', 'self_top_right']:
        draw_self_loop(ax, src, label, mode=mode, active=active)
        return

    if mode == 'vertical_down_right':
        # A -> B
        x = positions['A'][0] + 0.18
        y1 = positions['A'][1] - 0.44
        y2 = positions['B'][1] + 0.44
        arrow = FancyArrowPatch(
            (x, y1), (x, y2),
            connectionstyle='arc3,rad=0.0',
            arrowstyle='-|>',
            mutation_scale=16,
            lw=lw,
            color=color,
            zorder=2
        )
        ax.add_patch(arrow)
        ax.text(x + 0.20, (y1 + y2) / 2, label, fontsize=16, color=color,
                ha='center', va='center', zorder=8)
        return

    if mode == 'vertical_up_left':
        # B -> A
        x = positions['A'][0] - 0.18
        y1 = positions['B'][1] + 0.44
        y2 = positions['A'][1] - 0.44
        arrow = FancyArrowPatch(
            (x, y1), (x, y2),
            connectionstyle='arc3,rad=0.0',
            arrowstyle='-|>',
            mutation_scale=16,
            lw=lw,
            color=color,
            zorder=2
        )
        ax.add_patch(arrow)
        ax.text(x - 0.20, (y1 + y2) / 2, label, fontsize=16, color=color,
                ha='center', va='center', zorder=8)
        return

    x1, y1 = positions[src]
    x2, y2 = positions[dst]

    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        connectionstyle='arc3,rad=0.0',
        arrowstyle='-|>',
        mutation_scale=16,
        lw=lw,
        color=color,
        shrinkA=30,
        shrinkB=30,
        zorder=2
    )
    ax.add_patch(arrow)

    if mode == 'diag_up':
        tx, ty = ((x1 + x2) / 2 - 0.02, (y1 + y2) / 2 + 0.18)
    elif mode == 'diag_down':
        tx, ty = ((x1 + x2) / 2 - 0.02, (y1 + y2) / 2 - 0.22)
    else:  # right
        tx, ty = ((x1 + x2) / 2 + 0.02, (y1 + y2) / 2 + 0.18)

    ax.text(tx, ty, label, fontsize=16, color=color, ha='center', va='center', zorder=8)


def render_fsm(current_state='S', active_transition=None, title='Diagram FSD / FSM'):
    fig, ax = plt.subplots(figsize=(8.2, 4.9))
    fig.patch.set_facecolor('#eeeeee')
    ax.set_facecolor('#eeeeee')

    sx, sy = positions['S']

    # start arrow kiri
    start_arrow = FancyArrowPatch(
        (-0.92, sy), (sx - 0.43, sy),
        arrowstyle='-|>',
        mutation_scale=16,
        lw=1.6,
        color='#304ffe',
        zorder=1
    )
    ax.add_patch(start_arrow)

    # edges
    for src, dst, label, mode in edge_specs:
        active = False
        if active_transition is not None:
            a_src, a_label, a_dst = active_transition
            if (src, dst) == (a_src, a_dst):
                if label == a_label or (src == 'C' and dst == 'C' and a_label in ['0', '1']):
                    active = True
        draw_edge(ax, src, dst, label, mode=mode, active=active)

    # states
    for name, pos in positions.items():
        draw_state(ax, name, pos, active=(name == current_state), final=(name == FINAL_STATE))

    ax.set_xlim(-1.05, 6.45)
    ax.set_ylim(-2.25, 2.25)
    ax.axis('off')
    plt.show()

In [ ]:
input_box = widgets.Text(
    value='10101',
    description='Masukkan string:',
    placeholder='contoh: 10101',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px')
)

run_btn = widgets.Button(description='Mulai', button_style='success')
prev_btn = widgets.Button(description='Prev', button_style='info', disabled=True)
next_btn = widgets.Button(description='Next', button_style='info', disabled=True)
reset_btn = widgets.Button(description='Reset', button_style='warning', disabled=True)

status_box = widgets.HTML()
diagram_out = widgets.Output()
trace_out = widgets.Output()

app_state = {
    'steps': [],
    'accepted': False,
    'result': '',
    'reason': '',
    'idx': 0,
    'text': '',
}

def update_buttons():
    has_steps = len(app_state['steps']) > 0
    prev_btn.disabled = (not has_steps) or app_state['idx'] <= 0
    next_btn.disabled = (not has_steps) or app_state['idx'] >= len(app_state['steps']) - 1
    reset_btn.disabled = not has_steps

def render_current():
    if not app_state['steps']:
        return

    step = app_state['steps'][app_state['idx']]
    current_state = step['to_state']
    active_transition = None

    if step['from_state'] is not None:
        active_transition = (step['from_state'], step['symbol'], step['to_state'])

    with diagram_out:
        clear_output(wait=True)
        render_fsm(
            current_state=current_state,
            active_transition=active_transition,
            title=f"Diagram FSD / FSM - Langkah {app_state['idx']}"
        )

    sisa_input = step['remaining'] if step['remaining'] != '' else 'kosong'
    simbol = '-' if step['symbol'] is None else step['symbol']

    info = f"""
    <div style='padding:10px 14px; border:1px solid #d0d0d0; border-radius:8px; margin-top:6px;'>
      <b>Langkah:</b> {app_state['idx']}<br>
      <b>Simbol:</b> {simbol}<br>
      <b>Current state:</b> {current_state}<br>
      <b>Keterangan state:</b> {state_desc[current_state]}<br>
      <b>Sisa input:</b> {sisa_input}<br>
      <b>Output:</b> {step['log']}
    </div>
    """

    final_info = ""
    if app_state['idx'] == len(app_state['steps']) - 1:
        warna = '#1a7f37' if app_state['accepted'] else '#b42318'
        hasil = 'DITERIMA' if app_state['accepted'] else 'DITOLAK'
        final_info = f"""
        <div style='margin-top:10px; padding:10px 14px; border:1px solid #d0d0d0; border-radius:8px;'>
          <b>Hasil akhir:</b> <span style='color:{warna}; font-weight:bold;'>{hasil}</span><br>
          <b>Status:</b> {app_state['result']}<br>
          <b>Alasan:</b> {app_state['reason']}
        </div>
        """

    status_box.value = info + final_info

    with trace_out:
        clear_output(wait=True)
        print('Jejak pembacaan:')
        for i, s in enumerate(app_state['steps'][:app_state['idx'] + 1]):
            tanda = '->' if i == app_state['idx'] else '  '
            print(f"{tanda} Langkah {i}: {s['log']}")

    update_buttons()

def on_run(_):
    text = input_box.value.strip()

    if len(text) == 0:
        status_box.value = (
            "<div style='color:#b42318; padding:10px 14px; border:1px solid #f0c0c0; border-radius:8px;'>"
            "String kosong tidak dikenali oleh FSD"
            "</div>"
        )
        with diagram_out:
            clear_output(wait=True)
            render_fsm(current_state='S', active_transition=None)
        with trace_out:
            clear_output(wait=True)
        app_state['steps'] = []
        update_buttons()
        return

    if not valid_binary(text):
        status_box.value = (
            "<div style='color:#b42318; padding:10px 14px; border:1px solid #f0c0c0; border-radius:8px;'>"
            "Input mengandung karakter selain 0 dan 1"
            "</div>"
        )
        with diagram_out:
            clear_output(wait=True)
            render_fsm(current_state='S', active_transition=None)
        with trace_out:
            clear_output(wait=True)
        app_state['steps'] = []
        update_buttons()
        return

    steps, accepted, result, reason = simulate_steps(text)
    app_state['steps'] = steps
    app_state['accepted'] = accepted
    app_state['result'] = result
    app_state['reason'] = reason
    app_state['idx'] = 0
    app_state['text'] = text
    render_current()

def on_prev(_):
    if app_state['idx'] > 0:
        app_state['idx'] -= 1
        render_current()

def on_next(_):
    if app_state['idx'] < len(app_state['steps']) - 1:
        app_state['idx'] += 1
        render_current()

def on_reset(_):
    if app_state['steps']:
        app_state['idx'] = 0
        render_current()

run_btn.on_click(on_run)
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
reset_btn.on_click(on_reset)

control_row = widgets.HBox([input_box, run_btn])
nav_row = widgets.HBox([prev_btn, next_btn, reset_btn])

display(widgets.VBox([
    control_row,
    nav_row,
    status_box,
    diagram_out,
    trace_out,
]))

with diagram_out:
    render_fsm(current_state='S', active_transition=None)

## Cara pakai
1. Jalankan semua cell.
2. Masukkan string biner, misalnya `10101`.
3. Klik **Mulai**.
4. Klik **Next** untuk maju satu langkah.
5. Klik **Prev** untuk kembali ke langkah sebelumnya.
6. Klik **Reset** untuk kembali ke state awal.
7. Pada langkah terakhir, notebook akan menampilkan apakah string dikenali atau tidak dikenali oleh FSD.
